### Intact animals performance curves

In [ ]:
from mab_abstract.mab_abstract_datagen import build_abstract_perf_tier

perf_df = build_abstract_perf_tier()

In [ ]:
import mab_subjects
from mab_abstract.mab_abstract_plotters import plot_tier_row, get_fig

df = mab_subjects.GroupData().abstract_perf_tier.latest
df_animal_8020 = df[
    (df["dataset"] != "RNNdataset")
    & (df["lesion"] == "intact")
    & (df["paradigm"] == "8020")
]
df_animal_8020["trial_id"] = df_animal_8020["trial_id"].astype(str)

fig = get_fig(nrows=9, ncols=6)

plot_tier_row(
    fig,
    row=0,
    df=df_animal_8020,
    y_cols=["perf", "perf_low_low", "perf_high_low", "perf_high_high"],
    ylabel="P(High)",
    ylim=(0.3, 0.9),
    ytick_step=0.1,
    stat_method="bootstrap",
    comparisons_correction="fdr_by",
    sep=0.8,
)

figpath = mab_subjects.FigPath.abstracts / "perf_intact"
fig.savefig(figpath)

### Intact animals switching curves

In [ ]:
from mab_abstract.mab_abstract_datagen import build_abstract_swp_tier

df_main = build_abstract_swp_tier(
    trial_filter=dict(min_trials=100, clip_max=30), trial_window=10
)

In [ ]:
import mab_subjects
from mab_abstract.mab_abstract_plotters import plot_tier_row, get_fig

df = mab_subjects.GroupData().abstract_swp_tier.latest

# Filter for animal data
animal_df = df[
    (df["dataset"] != "RNNdataset")
    & (df["lesion"] == "intact")
    & (df["paradigm"] == "8020")
]
animal_df["trial_id"] = animal_df["trial_id"].astype(str)

fig = get_fig(9, 6)

cats = ["all", "win", "lose"]
cat_labels = {"all": "P(Switch)", "win": "P(Switch | win)", "lose": "P(Switch | lose)"}

for c, cat in enumerate(cats):
    plot_tier_row(
        fig,
        row=c,
        df=animal_df,
        y_cols=[
            f"swp_{cat}",
            f"swp_{cat}_low_low",
            f"swp_{cat}_high_low",
            f"swp_{cat}_high_high",
        ],
        ylabel=cat_labels[cat],
        ylim=(0, 0.2),
        ytick_step=0.05,
        stat_method="cluster_permutation",
        sep=0.8,
    )

figpath = mab_subjects.FigPath.abstracts / "swp_intact"
fig.savefig(figpath)

### Tau comparisons intact animals

In [ ]:
from mab_abstract.mab_abstract_datagen import build_abstract_tau_tier

df_main = build_abstract_tau_tier(
    trial_filter=dict(min_trials=100, clip_max=100), model="double"
)

In [ ]:
import mab_subjects
from mab_abstract.mab_abstract_plotters import plot_tier_row, get_fig
from statplotannot.plots import SeabornPlotter
from mab_colors import Palette2Arm

df = mab_subjects.GroupData().abstract_tau_tier.latest
palette = Palette2Arm().as_dict()

# Filter for animal data
animal_df = df[
    (df["dataset"] != "RNNdataset")
    & (df["lesion"] == "intact")
    & (df["paradigm"] == "8020")
]
animal_df = animal_df[~animal_df["tau_type"].isin(["all", "high_high"])]

fig = get_fig(9, 7)
ax = fig.subplot(fig.gs[0])
SeabornPlotter(
    data=animal_df,
    x="tau_type",
    y="tau_value",
    hue="group",
    hue_order=["unstruc", "struc"],
    ax=ax,
).boxplot_filled(palette=palette).stat_bootstrap()

figpath = mab_subjects.FigPath.abstracts / "tau_intact"
fig.savefig(figpath)